In [0]:
%run ../notebooks/create_schema

In [0]:
%run ../src/extractor

In [0]:
%run ../src/landing_writer

In [0]:
%run ../src/bronze_writer

In [0]:
%run ../src/control_log

In [0]:

import yaml
from datetime import datetime, timezone

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

def reconciliar(qtd_lida, qtd_gravada, limiar):
    if qtd_lida == 0:
        return "SUCCESS", None
    divergencia = abs(qtd_lida - qtd_gravada) / qtd_lida * 100
    if divergencia > limiar:
        return "PARTIAL", f"Divergência de {divergencia:.2f}% entre origem e destino"
    return "SUCCESS", None

extractor = MongoExtractor(config["database"])
landing = LandingWriter(dbutils, config["landing_base_path"])
bronze = BronzeWriter(spark, config["catalog"], config["schema"])
control = ControlLog(spark, config["control_table"])

for item in config["collections"]:
    collection, modo_carga, chave = item["collection"], item["modo_carga"], item.get("chave", "_id")
    campo_watermark = item.get("campo_watermark")

    start_time = datetime.now(timezone.utc)
    watermark_ini = control.get_last_watermark(collection) if modo_carga == "incremental" else None
    watermark_fim = watermark_ini
    qtd_lida = qtd_gravada = 0
    status, erro = "SUCCESS", None

    try:
        for lote in extractor.extract(collection, modo_carga, campo_watermark, watermark_ini,
                                       item.get("campos"), item.get("batch_size", 2000)):
            qtd_lida += len(lote)

            file_path = landing.write(lote, collection, chave)
            qtd_gravada += bronze.write_from_landing(file_path, collection, modo_carga)

            if campo_watermark:
                valores = [d[campo_watermark] for d in lote if campo_watermark in d]
                watermark_fim = max(valores, default=watermark_fim)

        status, erro = reconciliar(qtd_lida, qtd_gravada, config["reconciliation_threshold_pct"])
    except Exception as e:
        status, erro = "FAILED", str(e)

    control.registrar(collection, modo_carga, watermark_ini, watermark_fim,
                       qtd_lida, qtd_gravada, start_time, datetime.now(timezone.utc), status, erro)
    print(f"{collection}: {status} | lido={qtd_lida} gravado={qtd_gravada}")